# 03 — Preprocessing

Reuse the production split + scale code from `prices.data.preprocessing` so the notebook stays in sync with what the trainer and the Flask app actually do.

Outputs (written to `notebooks/_artifacts/`):
- `X_train_scaled.npy`, `X_test_scaled.npy`
- `y_train.npy`, `y_test.npy`
- `feature_names.json`
- `scaler.joblib`

In [1]:
import json
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parents[1]
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

SCRATCH = REPO_ROOT / "notebooks" / "_artifacts"

import numpy as np
import pandas as pd
from joblib import dump

import config
from prices.data.preprocessing import scale, split_data

df = pd.read_parquet(SCRATCH / "clean.parquet")
print(f"loaded {df.shape[0]:,} rows")

loaded 9,997 rows


## Split

In [2]:
X_train, X_test, y_train, y_test = split_data(
    df,
    test_ratio=config.TEST_SIZE,
    random_state=config.SEED,
)
print(f"train rows: {len(X_train):>6,}")
print(f"test  rows: {len(X_test):>6,}")
print(f"train cols: {list(X_train.columns)}")

train rows:  7,997
test  rows:  2,000
train cols: ['year', 'age', 'beds', 'baths', 'home_size', 'parcel_size', 'pool', 'dist_cbd', 'dist_lakes', 'x_coord', 'y_coord']


## Scale

Standardize features (zero mean, unit variance) using a `StandardScaler` fit on train only.

In [3]:
X_train_scaled, X_test_scaled, scaler = scale(X_train, X_test)

summary = pd.DataFrame({
    "feature": X_train.columns,
    "raw_mean": X_train.mean().values,
    "raw_std": X_train.std().values,
    "scaled_mean": X_train_scaled.mean(axis=0),
    "scaled_std": X_train_scaled.std(axis=0),
})
summary

,feature,raw_mean,raw_std,scaled_mean,scaled_std
0,year,2.002341e+03,1.533930,-5.393177e-14,1.0
1,age,1.310079e+01,14.450614,2.843237e-17,1.0
2,beds,3.360635e+00,0.777041,1.039559e-16,1.0
3,baths,2.222021e+00,0.723049,2.772156e-16,1.0
4,home_size,1.922261e+03,812.705459,1.137295e-16,1.0
5,parcel_size,1.057900e+04,15477.441229,-2.665535e-18,1.0
6,pool,2.167063e-01,0.412027,1.332767e-17,1.0
7,dist_cbd,1.450375e+04,6209.678087,-2.558913e-16,1.0
8,dist_lakes,1.491969e+03,1761.491964,1.048444e-16,1.0
9,x_coord,5.329689e+05,40858.262837,1.411401e-15,1.0


## Persist for downstream notebooks

In [4]:
np.save(SCRATCH / "X_train_scaled.npy", X_train_scaled)
np.save(SCRATCH / "X_test_scaled.npy", X_test_scaled)
np.save(SCRATCH / "y_train.npy", y_train.to_numpy())
np.save(SCRATCH / "y_test.npy", y_test.to_numpy())
(SCRATCH / "feature_names.json").write_text(json.dumps(list(X_train.columns)))
dump(scaler, SCRATCH / "scaler.joblib")
X_test.to_parquet(SCRATCH / "X_test_raw.parquet", index=False)

for f in sorted(SCRATCH.iterdir()):
    print(f"{f.name:<25} {f.stat().st_size:>10,} bytes")

X_test_raw.parquet            80,871 bytes
X_test_scaled.npy            176,128 bytes
X_train_scaled.npy           703,864 bytes
clean.parquet                423,335 bytes
feature_names.json               116 bytes
scaler.joblib                  1,183 bytes
y_test.npy                    16,128 bytes
y_train.npy                   64,104 bytes
